# SSI FastConnect — kiểm tra kết nối chỉ đọc

Notebook này tải đúng nhánh `feature/vietnam-market-foundation` của TradingAgents và chạy bài kiểm tra REST đã sửa định dạng ngày. Nó chỉ đọc OHLCV, không có chức năng đặt lệnh.

**Bảo mật:** notebook thử lấy `SSI_API_KEY` và `SSI_API_SECRET` từ Colab Secrets trước; nếu chưa có thì nhập ẩn trong ô tiếp theo. Không dán key vào code và không chia sẻ notebook kèm credentials.

In [ ]:
import os, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/PhamHuuQan/TradingAgents.git"
BRANCH = "feature/vietnam-market-foundation"
REPO_DIR = Path("/content/TradingAgents")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--depth", "1", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "-B", BRANCH, "FETCH_HEAD"], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "requests", "python-dotenv"], check=True)
sys.path.insert(0, str(REPO_DIR))
print("Repository branch and minimal dependencies ready.")

In [ ]:
from datetime import datetime, timedelta
from getpass import getpass
from zoneinfo import ZoneInfo

def read_colab_secret(name):
    try:
        from google.colab import userdata
        value = userdata.get(name)
    except Exception:
        value = None
    return value.strip() if isinstance(value, str) and value.strip() else None

api_key = read_colab_secret("SSI_API_KEY") or getpass("SSI_API_KEY (input hidden): ")
api_secret = read_colab_secret("SSI_API_SECRET") or getpass("SSI_API_SECRET (input hidden): ")
if not api_key.strip() or not api_secret.strip():
    raise ValueError("Both SSI credentials are required; no credentials were saved to the notebook.")

os.environ["SSI_API_KEY"] = api_key.strip()
os.environ["SSI_API_SECRET"] = api_secret.strip()
del api_key, api_secret

today_vn = datetime.now(ZoneInfo("Asia/Ho_Chi_Minh")).date()
from_date = (today_vn - timedelta(days=30)).isoformat()
to_date = today_vn.isoformat()
print(f"Running read-only SSI smoke test: FPT, {from_date} to {to_date}.")

from tradingagents.dataflows.vietnam_market.smoke import main
exit_code = main(["--symbol", "FPT", "--from-date", from_date, "--to-date", to_date, "--page-size", "100"])
if exit_code != 0:
    raise RuntimeError("SSI REST smoke test failed. Read the sanitized error above; do not share credentials.")

os.environ.pop("SSI_API_KEY", None)
os.environ.pop("SSI_API_SECRET", None)

## Cách đọc kết quả

- `SSI read-only connection: OK` và số nến lớn hơn 0: xác thực, REST request và chuẩn hóa dữ liệu đã chạy được.
- `HTTP 400`: ghi lại status/code và thông báo đã che thông tin nhạy cảm; không chụp hoặc gửi key/secret.
- `HTTP 401/403`: kiểm tra quyền Market Data và trạng thái cặp key/secret trong SSI Developer Portal.

Bài kiểm tra này xác nhận kết nối và dữ liệu trả về; dữ liệu lịch sử vừa tải không tự động trở thành dữ liệu point-in-time hợp lệ cho backtest.